# Bulan 4: Pengujian & Perbandingan Model

Di tahap ini kita bakal nyobain beberapa model Machine Learning buat klasifikasi teks dari dataset yang udah dilabeli di Bulan 3. Tujuannya buat nyari model mana yang paling cocok buat case kita.

**Model yang diuji:**
1. Naive Bayes (MultinomialNB)
2. Support Vector Machine (SVM)
3. Logistic Regression
4. Random Forest

Karena data kita imbalanced (kategori 'Netral' jauh lebih banyak), kita bakal coba dua skenario:
- **Skenario A (Binary):** Netral vs Ada Keluhan (gabungan semua kategori non-Netral)
- **Skenario B (Multi-class):** Semua kategori (Insomnia, Depresi, Cemas, Stress, Netral)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score
import warnings
warnings.filterwarnings('ignore')

# Baca dataset hasil labeling Bulan 3
df = pd.read_excel('../Bulan 3/dataset_labeled_bulan3.xlsx')

# Pake kolom normalized_text kalo ada, kalo nggak pake cleaned_text
text_col = 'normalized_text' if 'normalized_text' in df.columns else 'cleaned_text'

# Buang baris yang teksnya kosong
df = df.dropna(subset=[text_col])

print(f'Total data: {len(df)} baris')
print(f'\nDistribusi label awal:')
print(df['Kategori_Mental'].value_counts())

---
## Skenario A: Binary Classification (Netral vs Ada Keluhan)
Karena data non-Netral jumlahnya sedikit, kita gabungin dulu jadi satu kelas: **Ada Keluhan**.

In [ ]:
# Bikin label binary
df['label_binary'] = df['Kategori_Mental'].apply(
    lambda x: 'Netral' if x == 'Lainnya / Netral' else 'Ada Keluhan'
)

print('Distribusi Binary:')
print(df['label_binary'].value_counts())

In [ ]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=3000)
X = tfidf.fit_transform(df[text_col])
y_binary = df['label_binary']

# Split data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

print(f'Train: {X_train.shape[0]} baris')
print(f'Test : {X_test.shape[0]} baris')

In [ ]:
# Definisi model-model yang mau dibandingin
models = {
    'Naive Bayes': MultinomialNB(),
    'SVM (LinearSVC)': LinearSVC(max_iter=5000, class_weight='balanced'),
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
}

# Training dan evaluasi tiap model
results_binary = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    # Cross-validation (5-fold) buat ngukur stabilitas model
    cv_scores = cross_val_score(model, X, y_binary, cv=5, scoring='f1_weighted')
    
    results_binary.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'F1-Score (Weighted)': round(f1, 4),
        'CV Mean F1': round(cv_scores.mean(), 4),
        'CV Std': round(cv_scores.std(), 4)
    })
    
    print(f'\n=== {name} ===')
    print(classification_report(y_test, y_pred))

In [ ]:
# Tabel perbandingan model (Binary)
df_results_binary = pd.DataFrame(results_binary)
df_results_binary = df_results_binary.sort_values('F1-Score (Weighted)', ascending=False)

print('TABEL PERBANDINGAN MODEL - SKENARIO BINARY (Netral vs Ada Keluhan)')
print('=' * 70)
display(df_results_binary)

---
## Skenario B: Multi-class Classification
Sekarang kita coba klasifikasi ke semua kategori yang ada (Insomnia, Depresi, Cemas, Stress, Netral).

In [ ]:
y_multi = df['Kategori_Mental']

X_train_m, X_test_m, y_train_m, y_test_m = train_test_split(
    X, y_multi, test_size=0.2, random_state=42, stratify=y_multi
)

# Training dan evaluasi tiap model (multi-class)
results_multi = []

for name, model in models.items():
    # Buat instance baru biar nggak pake model yang udah di-train binary
    if name == 'Naive Bayes':
        m = MultinomialNB()
    elif name == 'SVM (LinearSVC)':
        m = LinearSVC(max_iter=5000, class_weight='balanced')
    elif name == 'Logistic Regression':
        m = LogisticRegression(max_iter=1000, class_weight='balanced')
    else:
        m = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    
    m.fit(X_train_m, y_train_m)
    y_pred_m = m.predict(X_test_m)
    
    acc = accuracy_score(y_test_m, y_pred_m)
    f1 = f1_score(y_test_m, y_pred_m, average='weighted')
    
    cv_scores = cross_val_score(m, X, y_multi, cv=5, scoring='f1_weighted')
    
    results_multi.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'F1-Score (Weighted)': round(f1, 4),
        'CV Mean F1': round(cv_scores.mean(), 4),
        'CV Std': round(cv_scores.std(), 4)
    })
    
    print(f'\n=== {name} ===')
    print(classification_report(y_test_m, y_pred_m, zero_division=0))

In [ ]:
# Tabel perbandingan model (Multi-class)
df_results_multi = pd.DataFrame(results_multi)
df_results_multi = df_results_multi.sort_values('F1-Score (Weighted)', ascending=False)

print('TABEL PERBANDINGAN MODEL - SKENARIO MULTI-CLASS')
print('=' * 70)
display(df_results_multi)

---
## Ringkasan & Kesimpulan
Gabungin hasil kedua skenario buat nentuin model terbaik.

In [ ]:
print('=' * 70)
print('RINGKASAN PERBANDINGAN MODEL')
print('=' * 70)

print('\n--- Skenario A: Binary (Netral vs Ada Keluhan) ---')
best_binary = df_results_binary.iloc[0]
print(f"Model terbaik: {best_binary['Model']}")
print(f"  Accuracy   : {best_binary['Accuracy']}")
print(f"  F1-Score   : {best_binary['F1-Score (Weighted)']}")
print(f"  CV Mean F1 : {best_binary['CV Mean F1']} (+/- {best_binary['CV Std']})")

print('\n--- Skenario B: Multi-class ---')
best_multi = df_results_multi.iloc[0]
print(f"Model terbaik: {best_multi['Model']}")
print(f"  Accuracy   : {best_multi['Accuracy']}")
print(f"  F1-Score   : {best_multi['F1-Score (Weighted)']}")
print(f"  CV Mean F1 : {best_multi['CV Mean F1']} (+/- {best_multi['CV Std']})")

print('\n' + '=' * 70)
print('CATATAN:')
print('- Accuracy tinggi bukan berarti model bagus kalo datanya imbalanced.')
print('- Fokus utama liat F1-Score (Weighted) dan CV Mean F1.')
print('- CV Std rendah = model stabil (hasilnya konsisten tiap fold).')
print('- class_weight=balanced dipake biar model nggak bias ke kelas mayoritas.')
print('=' * 70)

In [ ]:
# Simpan hasil perbandingan ke Excel
with pd.ExcelWriter('hasil_perbandingan_model.xlsx') as writer:
    df_results_binary.to_excel(writer, sheet_name='Binary', index=False)
    df_results_multi.to_excel(writer, sheet_name='Multi-class', index=False)

print('Hasil perbandingan disimpan di: hasil_perbandingan_model.xlsx')